In [1]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceM4/FineVisionMax", split="train", streaming=True)
print(next(iter(dataset)))

/home/ankitanand/Documents/pp/VLM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'images': [<PIL.PngImagePlugin.PngImageFile image mode=RGB size=1506x1102 at 0xF88088DAA790>], 'texts': [{'user': 'What is the primary key of the Equipment entity?', 'assistant': 'The primary key of the Equipment entity is EquipmentID, which uniquely identifies each piece of equipment in the diagram.\nAnswer: EquipmentID'}, {'user': 'What is the relationship between Equipment and Maintenance Record?', 'assistant': 'The relationship between Equipment and Maintenance Record is labeled as "Maintained By", indicating that one piece of equipment can have multiple maintenance records associated with it.\nAnswer: Maintained By'}, {'user': 'How many relationships does the Supplier entity have?', 'assistant': 'The Supplier entity has two relationships: "Supplied By" with Equipment and "Delivered By" with Logistics Company, indicating it serves as a midpoint in both flows.\nAnswer: 2'}, {'user': 'What is the cardinality of the relationship between Equipment and Supplier?', 'assistant': 'The car

In [2]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoProcessor, AutoTokenizer, AutoModelForCausalLM

class VisionLanguageModel(nn.Module):
    def __init__(self, vision_ckpt, language_ckpt, modality_input_dim=768, modality_output_dim=576):
        super().__init__()
        # only take vision backbone
        self.vision_encoder = AutoModel.from_pretrained(vision_ckpt).vision_model
        self.vision_processor = AutoProcessor.from_pretrained(vision_ckpt)
        self.modality_projector = nn.Linear(modality_input_dim, modality_output_dim, bias=False)
        self.tokenizer = AutoTokenizer.from_pretrained(language_ckpt)
        self.llm = AutoModelForCausalLM.from_pretrained(language_ckpt)

    def forward(self, text, image):
        processed_img = self.vision_processor(images=[image], return_tensors="pt").to(self.llm.device)
        image_embd = self.vision_encoder(**processed_img).last_hidden_state
        # match llm dtype
        image_embd = self.modality_projector(image_embd).to(dtype=self.llm.dtype)
        input_ids = self.tokenizer(text, return_tensors="pt").input_ids.to(self.llm.device)

        token_embd = self.llm.model.embed_tokens(input_ids)
        combined_embd = torch.cat((image_embd, token_embd), dim=1)
        # Concatenate image embeddings to token embeddings

        logits = self.llm(inputs_embeds=combined_embd).logits
        shift_logits = logits[:, image_embd.size(1):-1, :].contiguous()
        # remove images
        shift_labels = input_ids[:, 1:].contiguous()

        loss = nn.functional.cross_entropy(shift_logits.reshape(-1,
                                   shift_logits.size(-1)),
                                   shift_labels.reshape(-1))

        return logits, loss

In [3]:
vision_encoder_ckpt = "google/siglip2-base-patch16-256"
language_model_ckpt = "HuggingFaceTB/SmolLM2-135M-Instruct"

vlm = VisionLanguageModel(vision_encoder_ckpt, language_model_ckpt).to("cuda")

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 2981.59it/s]


In [4]:
print(vlm.vision_encoder.config.hidden_size)   # SigLIP2 base → 768
print(vlm.llm.config.hidden_size)              # SmolLM2-135M → 576

768
576


In [5]:
import transformers, inspect
print(transformers.__version__)
print(transformers.AutoTokenizer, inspect.getfile(transformers.AutoTokenizer))

5.15.0
<class 'transformers.models.auto.tokenization_auto.AutoTokenizer'> /home/ankitanand/Documents/pp/VLM/.venv/lib/python3.11/site-packages/transformers/models/auto/tokenization_auto.py


In [7]:
sample = next(iter(dataset))

def apply_chat_template_to_sample(sample):
    messages_list = []
    for data_dict in sample['texts']:
        messages_list.append({'role': 'user',
                              'content': data_dict['user']})
        messages_list.append({'role': 'assistant',
                              'content': data_dict['assistant']})
    return vlm.tokenizer.apply_chat_template(messages_list,
                                             tokenize=False)

text = apply_chat_template_to_sample(sample)

In [ ]:
# import torch.optim as optim

# optimizer = optim.Adam(vlm.parameters(), lr=1e-5)
# step = 0    
# for sample in dataset:
#     text = apply_chat_template_to_sample(sample)
#     if len(text) > 3000 or len(sample['images']) != 1:
#         continue
#     images = [sample['images'][0].convert('RGB')]
#     step += 1

#     optimizer.zero_grad()
#     logits, loss = vlm(text, images)

#     loss.backward()
#     optimizer.step()

#     print(f"step {step} | loss {loss.item():.4f}")

In [11]:
dataset = load_dataset("HuggingFaceM4/FineVisionMax", split="train", streaming=True)
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

def tokenize_sample(sample):
    messages_list = []
    for data_dict in sample['texts']:
        messages_list.append({'role': 'user', 'content': data_dict['user']})
        messages_list.append({'role': 'assistant', 'content':data_dict['assistant']})
        tokens = tokenizer.apply_chat_template(messages_list, tokenize=True, return_dict=False)
    return tokens

max_length = 2048
curr_sample = []
batch = []
max_batch_size = 8  

for sample in dataset:
    tokens = tokenize_sample(sample)
    if len(tokens) > max_length:
        continue # skip samples that are two long
    if len(curr_sample) + len(tokens) < max_length:
        curr_sample.extend(tokens)
    else:
        if len(batch) == max_batch_size:
            break # We've reached the max batch size, so we need to break   
        batch.append(curr_sample)
        curr_sample = tokens

## Naive Greedy knapsack algorithm

In [12]:
pool_tokens = []
pool_size = 50
for sample in dataset:
    toks = tokenize_sample(sample)
    if len(toks) <= max_length:
        pool_tokens.append(toks)
        if len(pool_tokens) >= pool_size:
            break

pool_tokens.sort(key=len, reverse=True) # Sort by length descending

batch_tokens = []
for toks in pool_tokens:
    placed = False
    # Try to place in an existing row
    for row_idx, row in enumerate(batch_tokens):
        curr_len = sum(len(seg) for seg in row)
        if curr_len + len(toks) <= max_length:
            batch_tokens[row_idx].append(toks)
            placed = True
            break   

    if placed:
        continue
    # Could not place in any existing row, start a new row if possible
    batch_tokens.append([toks])

## Balanced Greedy Knapsack
https://arxiv.org/abs/2501.14818

In [13]:
def balanced_greedy_knapsack(samples, L):
   # Step 1: Sort the samples
    samples.sort(key=len, reverse=True)
    total_length = sum([len(sample) for sample in samples])
    min_knapsacks = (total_length + L - 1) // L
    # Step 2: Initialize knapsacks
    knapsacks=[[] for _ in range(min_knapsacks)]
    knapsack_lengths = [0] * min_knapsacks
    # Step 3: Distribute samples across knapsacks
    ks_index = 0
    sample_index = 0
    while sample_index < len(samples):
        length = len(samples[sample_index])
        if knapsack_lengths[ks_index]+length<=L:
            knapsacks[ks_index].append(length)
            knapsack_lengths[ks_index] += length
            sample_index += 1
        else:
            knapsacks.append([])
            knapsack_lengths.append(0)
        ks_index = np.argmin(knapsack_lengths)
    return knapsacks